# Fallos en Placas de Acero (Steel Plates Faults)


- Este dataset está disponible en Kaggle (originalmente donado por Semeion Research Center).

- No contiene imágenes crudas, sino características extraídas mediante visión artificial de imágenes de placas de acero.

- **Nombre:** Steel Plates Faults.

- **Contexto:** Ingeniería Metalúrgica / Control de Calidad.

- **Objetivo:** Clasificar el tipo de defecto superficial en una placa de acero inoxidable basándose en características geométricas y radiométricas.

### Las Variables Predictoras ($X$) - 27 Características

#### Geometría y Forma:
- Pixels_Areas, Perimeter: Tamaño del defecto.
- X_Perimeter, Y_Perimeter: Proyecciones del perímetro.
- Shape_Index, Square_Index: ¿El defecto es redondo? ¿Es cuadrado?
- Steel_Plate_Thickness: Grosor de la placa (dato de proceso).
  
#### Posición (Ubicación espacial):
- X_Minimum, X_Maximum: Dónde está el defecto respecto al ancho de la banda transportadora.
- Y_Minimum, Y_Maximum: Posición longitudinal.

#### Luminosidad y Textura (Radiometría):
- Sum_of_Luminosity, Minimum_of_Luminosity, Maximum_of_Luminosity: Intensidad de luz reflejada. Ayuda a distinguir, por ejemplo, una mancha de aceite (oscura) de un rasguño (brillante).

#### Bordes y Contornos:
- Edges_Index, Empty_Index: Complejidad del borde del defecto.

### La Variable Objetivo ($y$) - 7 Tipos de Fallos

Las clases que vamos a predecir son fallos típicos en laminación:
- **Pastry:** Defectos por material pegado o residuos aplastados.
- **Z_Scratch:** Arañazos en zigzag.K_Scatch:
- **Arañazos** longitudinales específicos.
- **Stains:** Manchas (líquidos, óxido).
- **Dirtiness:** Suciedad superficial.
- **Bumps:** Protuberancias o abolladuras.
- **Other_Faults:** Cajón de sastre para otros defectos.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay


In [2]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv('SteelPlatesFaults.csv')

In [3]:
# Carga del dataset
#df = pd.read_csv('data/SteelPlatesFaults.csv')
df.head()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


In [4]:
# PREPARACIÓN DEL TARGET (y) y de data (X)

# Lista de las 7 columnas que representan los tipos de fallos
target_cols = [
    'Pastry', 'Z_Scratch', 'K_Scatch', 'Stains', 
    'Dirtiness', 'Bumps', 'Other_Faults'
]
# Usamos .idxmax(): Mira las 7 columnas y devuelve el NOMBRE de la columna que tiene el valor más alto (1).
# Convierte [1, 0, 0, ...] -> "Pastry"
y_nombres = df[target_cols].idxmax(axis=1)
# Convertimos los nombres a números (0 a 6)
encoder = LabelEncoder()
y = encoder.fit_transform(y_nombres)
X = df.drop(columns=target_cols)

In [5]:
# DIVISIÓN TRAIN / TEST 

# Separamos el 20% para test. Usamos stratify=y para mantener la proporción de fallos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de y_train:", y_train.shape)
y

Dimensiones de X_train: (1552, 27)
Dimensiones de y_train: (1552,)


array([4, 4, 4, ..., 3, 3, 3])